In [ ]:
import UQ_Toolbox as uq  # ← Changed
from medMNIST.utils import train_load_datasets_resnet as tr
import torch
import numpy as np
import torchvision.transforms as transforms
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, roc_auc_score, accuracy_score, recall_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from torchvision import transforms


size = 224  # Image size for the models
batch_size = 4000  # Batch size for the DataLoader
model_global_perfs = {}
flag = 'breastmnist'
calib_method = 'platt'
color = False
uq_method = 'GPS'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
aug_folder = f'/mnt/data/psteinmetz/computer_vision_code/code/UQ_Toolbox/medMNIST/gps_augment/{size}*{size}/{flag}_calibration_set'

print(f"Processing {flag} with color={color}")
if color is True:
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[.5, .5, .5], std=[.5, .5, .5])
    ])
    
    transform_tta = transforms.Compose([
        transforms.ToTensor()
    ])
else:
    # For grayscale images, repeat the single channel to make it compatible with ResNet
    # ResNet expects 3 channels, so we repeat the single channel image
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
        transforms.Normalize(mean=[.5], std=[.5])
    ])
    
    transform_tta = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.repeat(3, 1, 1))
        ])
    
    
models = tr.load_models(flag, device=device)
[study_dataset_plain, calibration_dataset, test_dataset], [train_loader, calibration_loader, test_loader], info = tr.load_datasets(flag, color, size, transform, batch_size, cache_test=False)
train_loaders, val_loaders = tr.CV_train_val_loaders(None, study_dataset_plain, batch_size=batch_size)
task_type = info['task']  # Determine the task type (binary-class or multi-class)
num_classes = len(info['label'])  # Number of classes
[_, calibration_dataset_tta, test_dataset_tta], [_, calibration_loader_tta, test_loader_tta], _ = tr.load_datasets(flag, color, size, transform_tta, batch_size)

results = tr.evaluate_model(models, test_loader, data_flag=flag, device=None, output_dir=None, prefix="test", display_cm=True)
y_true, y_scores, digits, correct_idx, incorrect_idx, indiv_scores = uq.evaluate_models_on_loader(models, test_loader, device)
results_calib = tr.evaluate_model(models, calibration_loader, data_flag=flag, device=None, output_dir=None, prefix="calib", display_cm=True)
y_true_calib, y_scores_calib, digits_calib, correct_idx_calib, incorrect_idx_calib, indiv_scores_calib = uq.evaluate_models_on_loader(models, calibration_loader, device)


In [ ]:
def find_best_threshold_and_compute_metrics(values, correct_predictions, optimization_metric='balanced_accuracy'):
    """
    Find the best threshold and compute metrics.

    Args:
        values (numpy.ndarray): UQ values.
        correct_predictions (list): Indices of correct predictions.
        optimization_metric (str): Metric to optimize ('balanced_accuracy', 'sensitivity', 'specificity').

    Returns:
        None
    """
    # Function to compute metrics
    def compute_metrics(uq_values, labels, threshold):
        predictions = (uq_values <= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(labels, predictions).ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        balanced_acc = balanced_accuracy_score(labels, predictions)
        return tn, fp, fn, tp, specificity, sensitivity, balanced_acc

    # Update the function to handle ties in specificity or sensitivity
    def find_optimal_threshold(uq_values, labels, metric):
        thresholds = np.linspace(min(uq_values), max(uq_values), 1000)
        best_threshold = thresholds[0]
        best_metric_value = 0
        best_secondary_metric_value = 0

        for threshold in thresholds:
            _, _, _, _, specificity, sensitivity, balanced_acc = compute_metrics(uq_values, labels, threshold)

            if metric == 'balanced_accuracy' and balanced_acc > best_metric_value:
                best_metric_value = balanced_acc
                best_threshold = threshold
            elif metric == 'sensitivity':
                if sensitivity > best_metric_value or (sensitivity == best_metric_value and specificity > best_secondary_metric_value):
                    best_metric_value = sensitivity
                    best_secondary_metric_value = specificity
                    best_threshold = threshold
            elif metric == 'specificity':
                if specificity > best_metric_value or (specificity == best_metric_value and sensitivity > best_secondary_metric_value):
                    best_metric_value = specificity
                    best_secondary_metric_value = sensitivity
                    best_threshold = threshold

        return best_threshold

    # Find the optimal threshold
    labels = np.array([1 if i in correct_predictions else 0 for i in range(len(values))])
    optimal_threshold = find_optimal_threshold(values, labels, optimization_metric)

    # Compute the confusion matrix using the optimal threshold
    predictions = (values <= optimal_threshold).astype(int)
    cm = confusion_matrix(labels, predictions)
    
    # Display the confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 36}, xticklabels=['Failure', 'Success'], yticklabels=['Failure', 'Success'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

    # Print the optimal threshold and metrics
    _, _, _, _, specificity, sensitivity, balanced_acc = compute_metrics(values, labels, optimal_threshold)
    
    print(f"Optimal Threshold: {optimal_threshold}")
    print(f"Balanced Accuracy: {balanced_acc}")
    print(f"Specificity: {specificity}")
    print(f"Sensitivity: {sensitivity}")

    return balanced_acc

In [ ]:
def display_UQ_results(metric, correct_predictions, incorrect_predictions, y_axis_title, title, optim_metric, flag, swarmplot=False):
    balanced_acc = find_best_threshold_and_compute_metrics(metric, correct_predictions, optim_metric)
    fpr, tpr, auc = uq.roc_curve_UQ_method_computation([metric[k] for k in correct_predictions], [metric[j] for j in incorrect_predictions])
    uq.UQ_method_plot([metric[k] for k in correct_predictions], [metric[j] for j in incorrect_predictions], y_axis_title, title, flag, swarmplot)
    print(auc)
    return auc, balanced_acc

In [ ]:
best_aug = uq.perform_greedy_policy_search(
    aug_folder, correct_idx_calib, incorrect_idx_calib, num_workers=90, max_iterations=5, num_searches=30, top_k=3, plot=True, seed=42
)

Best GPS augmentations found: [['N2_M45___9_np.float64_30.459768168853316____0_np.float64_-11.915508202782874___.npz', 'N2_M45___6_np.float64_-9.460123380308211____13_np.float64_23.00332977870943___.npz', 'N2_M45___13_np.float64_-35.00157698110694____12_np.float64_10.885299181145754___.npz', 'N2_M45___3_np.float64_-42.05275662803854____9_np.float64_26.338886552574564___.npz', 'N2_M45___13_np.float64_43.762295821766614____13_np.float64_36.752834234644155___.npz'], ['N2_M45___12_np.float64_30.818425210682463____10_np.float64_1.7624678994916536___.npz', 'N2_M45___3_np.float64_-42.05275662803854____9_np.float64_26.338886552574564___.npz', 'N2_M45___12_np.float64_11.351661016859538____12_np.float64_-13.524167490309729___.npz', 'N2_M45___5_np.float64_-6.591905323410799____9_np.float64_1.713205914189281___.npz', 'N2_M45___11_np.float64_-30.06938264183928____9_np.float64_16.849338768000614___.npz'], ['N2_M45___9_np.float64_-31.680119058646717____3_np.float64_5.067230156444751___.npz', 'N2_M45___6_np.float64_-9.460123380308211____13_np.float64_23.00332977870943___.npz', 'N2_M45___13_np.float64_-37.376464525880934____12_np.float64_4.689131800927385___.npz', 'N2_M45___3_np.float64_-42.05275662803854____9_np.float64_26.338886552574564___.npz', 'N2_M45___13_np.float64_43.762295821766614____13_np.float64_36.752834234644155___.npz']]

In [ ]:
if isinstance(best_aug, list) and all(isinstance(policy, list) for policy in best_aug):
    transformation_pipeline = []
    for aug in best_aug:
        n, m, transformations = uq.extract_gps_augmentations_info(aug)
        transformation_pipeline.append(transformations)

In [ ]:
metric, global_preds_GPS = uq.TTA(
    transformation_pipeline, models, test_dataset, device,
    usingBetterRandAugment=True, n=n, m=m, nb_channels=3, image_size=size,
    image_normalization=True, mean=0.5, std=0.5, batch_size=batch_size
)

In [ ]:
auc, b_acc = display_UQ_results(metric, correct_idx, incorrect_idx, 'GPS', 'GPS', optim_metric='balanced_accuracy', swarmplot=False, flag=flag)

In [ ]:
auc